# CVE2Patch — Kernel Evaluation
## Audit documentale e scoring kernel con KernJC

Notebook indipendente dalla pipeline CVE2Patch v4.4. Verifica la coerenza dei CSV, ripete i filtri Network/Local, ricostruisce i conteggi e confronta il dataset con la Tabella 6 del paper KernJC (RAID 2024).

**Non scarica codice, non compila kernel, non avvia VM, non esegue PoC.** I risultati del paper sono evidenze di letteratura, non verifiche locali. La simulazione dei punti ambiente non è un risultato operativo. Nella fase 2 si importano esclusivamente gli esiti documentati della verifica locale; KernJC sostituisce Vulhub nella categoria ambiente (30 punti), senza modifiche ai restanti pesi.

Requisiti: Python 3.10+, pandas e Jupyter/VS Code. Collocare i due CSV nella cartella del notebook, nella sottocartella `upload` oppure nella struttura originale CVE2Patch/data/processed. Modificare i percorsi nella cella di configurazione se necessario.

Fonte benchmark: Ruan et al., *KernJC: Automated Vulnerable Environment Generation for Linux Kernel Vulnerabilities*, RAID 2024, Table 6, p. 400 (PDF p. 17), DOI [10.1145/3678890.3678891](https://doi.org/10.1145/3678890.3678891), [PDF ufficiale](https://raid2024.github.io/papers/raid2024-1.pdf). Fonte scoring: notebook CVE2Patch v4.4-simple-single-cve, sezioni 2 e 11.

In [1]:
from pathlib import Path
import pandas as pd
import json, hashlib

START_DATE = '2019-01-01'
END_DATE = '2026-11-17'
ATTACK_VECTORS = ['NETWORK', 'LOCAL']
MIN_SCORE = 60
# Impostare esplicitamente questi percorsi se i CSV sono altrove.
BASE_CSV = None
ENRICHED_CSV = None

def resolve_path(explicit, names):
    if explicit is not None:
        p = Path(explicit)
        if not p.is_file():
            raise FileNotFoundError(p)
        return p
    for root in [Path.cwd(), Path.cwd() / 'upload',
                 Path.cwd() / 'CVE2Patch/data/processed',
                 Path.cwd() / 'data/processed']:
        for name in names:
            p = root / name
            if p.is_file():
                return p
    raise FileNotFoundError(f'Impostare il percorso del CSV: {names}')

base_path = resolve_path(BASE_CSV, ['kernel_cves(1).csv', 'kernel_cves.csv'])
enriched_path = resolve_path(ENRICHED_CSV, ['kernel_enriched(1).csv', 'kernel_enriched.csv'])
base = pd.read_csv(base_path, low_memory=False)
df = pd.read_csv(enriched_path, low_memory=False)
print('Base:', base.shape, '| Arricchito:', df.shape)

Base: (19482, 17) | Arricchito: (19482, 63)


## 1. Controlli di integrità
Le colonne comuni devono essere identiche dopo il riallineamento per ID. I valori mancanti non vengono sostituiti con falsi negativi.

In [2]:
for name, frame in [('base', base), ('enriched', df)]:
    assert frame.cve_id.notna().all(), f'{name}: ID mancanti'
    assert frame.cve_id.is_unique, f'{name}: ID duplicati'
    assert frame.kernel.eq(True).all(), f'{name}: record non kernel'
assert set(base.cve_id) == set(df.cve_id), 'Gli insiemi CVE dei CSV differiscono'
pd.testing.assert_frame_equal(
    base.sort_values('cve_id').reset_index(drop=True),
    df[base.columns].sort_values('cve_id').reset_index(drop=True)
)
flags = ['has_vulhub','msf_compatible','edb_compatible','has_poc_reference',
         'has_detection_rule','edb_verified','falco_syscall','sigma_eligible','elastic_eligible']
for flag in flags:
    assert df[flag].notna().all() and pd.api.types.is_bool_dtype(df[flag]), flag
assert df.has_detection_rule.equals(df[['falco_syscall','sigma_eligible','elastic_eligible']].any(axis=1))
print('Integrità CSV: OK; colonne base coerenti, ID unici, unione detection coerente.')
print('Valori mancanti:', df[['cvss','cvss_version','attack_vector']].isna().sum().to_dict())

Integrità CSV: OK; colonne base coerenti, ID unici, unione detection coerente.
Valori mancanti: {'cvss': 3911, 'cvss_version': 3911, 'attack_vector': 3911}


## 2. Periodo e filtro Attack Vector
Il pool kernel nel notebook originale precede il filtro AV. Manteniamo gli AV mancanti in una categoria separata.

In [3]:
period = df.published.str[:10].between(START_DATE, END_DATE)
valid = df.loc[period & df.status.ne('Rejected')].copy()
av = valid.attack_vector.isin(ATTACK_VECTORS)
eligible = valid.loc[av].copy()
unknown = valid.loc[valid.attack_vector.isna()]
other = valid.loc[~av & valid.attack_vector.notna()]
assert len(valid) == len(eligible) + len(unknown) + len(other)
print('Totale periodo:', len(valid))
print('AV ammesso:', len(eligible), '| AV mancante:', len(unknown), '| Altro AV:', len(other))
print(valid.attack_vector.fillna('MISSING').value_counts().to_string())
print('\nDistribuzione per anno di pubblicazione:')
print(valid.published.str[:4].value_counts().sort_index().to_string())

Totale periodo: 19482
AV ammesso: 15122 | AV mancante: 3911 | Altro AV: 449
attack_vector
LOCAL               13827
MISSING              3911
NETWORK              1295
ADJACENT_NETWORK      371
PHYSICAL               75
ADJACENT                3

Distribuzione per anno di pubblicazione:
published
2019     250
2020     122
2021     157
2022     313
2023     309
2024    4363
2025    5681
2026    8287


## 3. Copertura documentale
Le fonti si sovrappongono: usare OR per l'unione. La presenza di metadati non dimostra la funzionalità di un test o di una regola.

In [4]:
EVIDENCE = ['msf_compatible', 'edb_compatible', 'has_poc_reference']
for frame in [valid, eligible]:
    frame['documented_evidence'] = frame[EVIDENCE].any(axis=1)
summary_flags = ['has_vulhub', 'has_metasploit', 'msf_compatible', 'has_exploitdb',
                 'edb_compatible','edb_verified','has_github','has_poc_reference',
                 'falco_syscall','sigma_eligible','elastic_eligible','has_detection_rule',
                 'documented_evidence']
coverage = pd.DataFrame({
    'kernel':valid[summary_flags].sum(),
    'network_local':eligible[summary_flags].sum()
})
coverage['kernel_pct'] = (100 * coverage.kernel / len(valid)).round(3)
coverage['network_local_pct'] = (100 * coverage.network_local / len(eligible)).round(3)
print(coverage.to_string())

                     kernel  network_local  kernel_pct  network_local_pct
has_vulhub                0              0       0.000              0.000
has_metasploit           17             17       0.087              0.112
msf_compatible           17             17       0.087              0.112
has_exploitdb            21             19       0.108              0.126
edb_compatible           21             19       0.108              0.126
edb_verified             11             11       0.056              0.073
has_github              365            347       1.874              2.295
has_poc_reference       117            112       0.601              0.741
falco_syscall             1              1       0.005              0.007
sigma_eligible            1              1       0.005              0.007
elastic_eligible          2              2       0.010              0.013
has_detection_rule        3              3       0.015              0.020
documented_evidence     135           

## 4. Confronto con il benchmark KernJC
Gli identificativi provengono dalla Tabella 6 (66 righe). L'intersezione è un controllo di letteratura; l'assenza dalla tabella non costituisce una prova di incompatibilità.

In [5]:
PAPER_IDS = ["CVE-2015-3636","CVE-2016-0728","CVE-2016-10150","CVE-2016-4557","CVE-2016-6187","CVE-2016-6516","CVE-2016-8655","CVE-2016-9793","CVE-2017-1000112","CVE-2017-10661","CVE-2017-11176","CVE-2017-15265","CVE-2017-15649","CVE-2017-16995","CVE-2017-17052","CVE-2017-17053","CVE-2017-17712","CVE-2017-18344","CVE-2017-2636","CVE-2017-5123","CVE-2017-6074","CVE-2017-7184","CVE-2017-7308","CVE-2017-7533","CVE-2017-8824","CVE-2017-8890","CVE-2018-10840","CVE-2018-12232","CVE-2018-12233","CVE-2018-18559","CVE-2018-5333","CVE-2018-6555","CVE-2019-15666","CVE-2019-6974","CVE-2020-14381","CVE-2020-14386","CVE-2020-16119","CVE-2020-25656","CVE-2020-25669","CVE-2020-27194","CVE-2020-27830","CVE-2020-28097","CVE-2020-28941","CVE-2020-8835","CVE-2021-20226","CVE-2021-22555","CVE-2021-22600","CVE-2021-26708","CVE-2021-27365","CVE-2021-33909","CVE-2021-34866","CVE-2021-3490","CVE-2021-3573","CVE-2021-41073","CVE-2021-4154","CVE-2021-42008","CVE-2021-43267","CVE-2022-0185","CVE-2022-0995","CVE-2022-1015","CVE-2022-1786","CVE-2022-24122","CVE-2022-25636","CVE-2022-32250","CVE-2022-34918","CVE-2023-32233"]
assert len(PAPER_IDS) == len(set(PAPER_IDS)) == 66
for frame in [valid, eligible]:
    frame['kernjc_paper_match'] = frame.cve_id.isin(PAPER_IDS)
    frame['kernjc_local_test_status'] = 'not_tested'
paper_match = eligible.kernjc_paper_match
evidence = eligible.documented_evidence
groups = pd.Series({
    'Entrambi': int((paper_match & evidence).sum()),
    'Solo evidenza documentale': int((~paper_match & evidence).sum()),
    'Solo benchmark KernJC': int((paper_match & ~evidence).sum()),
    'Nessuno dei due': int((~paper_match & ~evidence).sum())
})
assert groups.sum() == len(eligible)
print('Paper: 66 | corrispondenze totali:', int(valid.kernjc_paper_match.sum()))
print('Paper con AV ammesso:', int(paper_match.sum()))
print('Evidenza documentale con AV ammesso:', int(evidence.sum()))
print(groups.to_string())
print('Unione:', int((paper_match | evidence).sum()))
print('Fuori dal benchmark, AV ammesso:', int((~paper_match).sum()))
print('Verifiche locali completate:', int(eligible.kernjc_local_test_status.ne('not_tested').sum()))

Paper: 66 | corrispondenze totali: 34
Paper con AV ammesso: 33
Evidenza documentale con AV ammesso: 128
Entrambi                        17
Solo evidenza documentale      111
Solo benchmark KernJC           16
Nessuno dei due              14978
Unione: 144
Fuori dal benchmark, AV ammesso: 15089
Verifiche locali completate: 0


## 5. Metadati utili all'analisi delle patch
I riferimenti sono indicatori documentali, non una stima del successo di KernJC. I CPE esportati non conservano gli estremi espliciti degli intervalli NVD. Per una verifica futura occorre consultare i dati NVD completi e le fonti ufficiali delle patch.

In [6]:
def kernel_metadata(frame):
    result = pd.DataFrame(index=frame.index)
    result['kernel_cpe'] = frame.vulnerable_cpe.str.contains(
        'cpe:2.3:o:linux:linux_kernel:', regex=False, na=False)
    result['no_vulnerable_cpe'] = frame.vulnerable_cpe.eq('[]')
    result['stable_reference'] = frame.references.str.contains('git.kernel.org/stable/', regex=False, na=False)
    result['patch_tag'] = frame.reference_details.str.contains('"Patch"', regex=False, na=False)
    return result

metadata_all = kernel_metadata(valid)
metadata_av = kernel_metadata(eligible)
print(pd.DataFrame({'kernel':metadata_all.sum(), 'network_local':metadata_av.sum()}).to_string())

                   kernel  network_local
kernel_cpe          13198          12947
no_vulnerable_cpe    6277           2169
stable_reference    18247          13998
patch_tag           12943          12721


## 6. Riproduzione dello scoring v4.4 — analisi aggregata
Formula invariata rispetto al notebook originale; nessun nuovo ranking operativo. I 30 punti dell'ambiente dipendono da Vulhub nel metodo originale. Gli scenari alternativi sono esclusivamente controfattuali e NON attestano la presenza di ambienti verificati.

In [7]:
WEIGHTS = dict(vulhub=30,msf_min=30,msf_max=40,edb_verified=30,
               edb_unverified=25,poc_primary=20,edb_bonus=5,poc_bonus=5,detection_rules=20)
assert WEIGHTS['vulhub']+WEIGHTS['msf_max']+WEIGHTS['edb_bonus']+WEIGHTS['poc_bonus']+WEIGHTS['detection_rules']==100

def score_components(frame):
    msf, edb, poc = (frame[c] for c in EVIDENCE)
    rank = frame.msf_best_rank.fillna(0)
    s = pd.DataFrame(index=frame.index)
    s['environment'] = frame.has_vulhub.astype(int)*WEIGHTS['vulhub']
    s['msf'] = (WEIGHTS['msf_min']+(WEIGHTS['msf_max']-WEIGHTS['msf_min'])*rank/600).where(msf,0)
    s['edb'] = frame.edb_verified.map({True:WEIGHTS['edb_verified'],False:WEIGHTS['edb_unverified']}).where(~msf,WEIGHTS['edb_bonus']).where(edb,0)
    s['poc'] = (msf|edb).map({True:WEIGHTS['poc_bonus'],False:WEIGHTS['poc_primary']}).where(poc,0)
    s['detection'] = frame.has_detection_rule.astype(int)*WEIGHTS['detection_rules']
    s['total'] = s.sum(axis=1).round(2)
    assert s.total.between(0,100).all()
    return s

def summarize_score(frame):
    s = score_components(frame)
    without_env = s.total - s.environment
    paper_env = (frame.has_vulhub | frame.kernjc_paper_match).astype(int)*WEIGHTS['vulhub']
    return {
        'CVE':len(frame),
        'Score 0':int(s.total.eq(0).sum()),
        'Score positivo':int(s.total.gt(0).sum()),
        'Massimo score':float(s.total.max()),
        'Soglia attuale':int(s.total.ge(MIN_SCORE).sum()),
        'Ipotesi ambiente per i soli match paper':int((without_env+paper_env).ge(MIN_SCORE).sum()),
        'Ipotesi ambiente per tutte':int((without_env+WEIGHTS['vulhub']).ge(MIN_SCORE).sum())
    }

scores = pd.DataFrame({'kernel':summarize_score(valid),'network_local':summarize_score(eligible)})
print('Soglia:', MIN_SCORE)
print(scores.to_string())
print('\nDistribuzione valori score originale (AV ammesso):')
print(score_components(eligible).total.value_counts().sort_index().to_string())

Soglia: 60
                                          kernel  network_local
CVE                                      19482.0        15122.0
Score 0                                  19346.0        14993.0
Score positivo                             136.0          129.0
Massimo score                               65.0           65.0
Soglia attuale                               2.0            2.0
Ipotesi ambiente per i soli match paper      8.0            8.0
Ipotesi ambiente per tutte                  25.0           25.0

Distribuzione valori score originale (AV ammesso):
total
0.00     14993
20.00      100
25.00        4
30.00        8
38.33        3
40.00        2
41.67        1
43.33        2
45.00        1
46.67        2
48.33        2
50.00        2
65.00        2


## 7. Controlli di regressione
I valori di riferimento valgono per lo snapshot analizzato con la configurazione originale. Modificando i CSV o i parametri, i risultati possono cambiare. Gli scostamenti sono mostrati e non nascosti.

In [8]:
expected = {
    'kernel':19482,'av':15122,'documented_evidence':128,'paper_match_all':34,
    'paper_match_av':33,'overlap':17,'evidence_only':111,'paper_only':16,'neither':14978,
    'score0_all':19346,'threshold_all':2,'threshold_av':2,'hypothetical_all_av':25
}
observed = {
    'kernel':len(valid),'av':len(eligible),'documented_evidence':int(evidence.sum()),
    'paper_match_all':int(valid.kernjc_paper_match.sum()),'paper_match_av':int(paper_match.sum()),
    'overlap':int(groups['Entrambi']),'evidence_only':int(groups['Solo evidenza documentale']),
    'paper_only':int(groups['Solo benchmark KernJC']),'neither':int(groups['Nessuno dei due']),
    'score0_all':int(scores.loc['Score 0','kernel']),
    'threshold_all':int(scores.loc['Soglia attuale','kernel']),
    'threshold_av':int(scores.loc['Soglia attuale','network_local']),
    'hypothetical_all_av':int(scores.loc['Ipotesi ambiente per tutte','network_local'])
}
audit = pd.DataFrame({'atteso':expected,'osservato':observed})
audit['coerente'] = audit.atteso.eq(audit.osservato)
print(audit.to_string())
print('Controlli coincidenti:', int(audit.coerente.sum()), '/',len(audit))
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()
print('SHA256 base:', sha256(base_path))
print('SHA256 enriched:', sha256(enriched_path))

                     atteso  osservato  coerente
kernel                19482      19482      True
av                    15122      15122      True
documented_evidence     128        128      True
paper_match_all          34         34      True
paper_match_av           33         33      True
overlap                  17         17      True
evidence_only           111        111      True
paper_only               16         16      True
neither               14978      14978      True
score0_all            19346      19346      True
threshold_all             2          2      True
threshold_av              2          2      True
hypothetical_all_av      25         25      True
Controlli coincidenti: 13 / 13
SHA256 base: dbfc29d10c12f144927221e7de37591acf194ba589cc2979b5264d8e333867e5
SHA256 enriched: b7bfee0f824b75e580b5f976b82413939545a128cb18217c8937e60750afc660


## 8. Scoring kernel definitivo — stessi pesi della v4.4

La popolazione di valutazione è il sottoinsieme Network/Local con almeno una delle tre evidenze documentali: Metasploit compatibile, Exploit-DB compatibile o riferimento PoC già indicizzato.

**Un solo cambiamento rispetto alla formula originale:** per il kernel, i 30 punti ambiente dipendono da una verifica locale documentata di KernJC, non da Vulhub. Rimangono invariati pesi, bonus, regole detection non cumulative e soglia 60.

Stati ambienti: `not_tested` (nessuna prova), `confirmed` (ambiente verificato), `failed` (verifica negativa), `inconclusive` (evidenze insufficienti). Il match con il paper non equivale a `confirmed`. I punteggi per i casi non confermati sono provvisori e non determinano l'inclusione nel benchmark.

Se disponibili, importare gli esiti da un CSV locale `kernjc_environment_results.csv` con colonne `cve_id,environment_status,evidence_note`. Una conferma richiede una nota che faccia riferimento alla validazione documentata dell'ambiente (versione/configurazione e avvio). La validità sostanziale della nota richiede revisione umana. Questo notebook non esegue KernJC e non verifica la vulnerabilità né il funzionamento dei PoC.

Il criterio finale di inclusione richiede **insieme**: evidenza documentale, ambiente confermato e score >= MIN_SCORE. Senza esiti locali non esiste ancora un pool definitivo.

Eseguendo questa sezione si genera `kernel_candidates.csv` nella directory di lavoro di Jupyter. Il CSV contiene tutte le CVE del perimetro documentale (128 nello snapshot di riferimento), con metadati, punteggi e stati ambiente. Rieseguendo la cella si aggiorna il file.

In [ ]:
# Facoltativo: impostare il percorso del file di esiti documentati.
ENV_RESULTS_CSV = Path('kernjc_environment_results.csv')
STATUS_VALUES = {'not_tested', 'confirmed', 'failed', 'inconclusive'}
LOG_COLUMNS = ['cve_id','environment_status','evidence_note']

def apply_kernel_scoring(frame, result_log):
    # L'input è il dataset Network/Local; le evidenze mantengono la definizione v4.4.
    candidates = frame.loc[frame.documented_evidence].copy()
    assert candidates.cve_id.is_unique
    missing = set(LOG_COLUMNS) - set(result_log.columns)
    if missing:
        raise ValueError(f'Colonne mancanti nel file esiti: {sorted(missing)}')
    results = result_log[LOG_COLUMNS].copy()
    if results.cve_id.isna().any() or not results.cve_id.is_unique:
        raise ValueError('Gli ID nel file esiti devono essere presenti e univoci.')
    if not results.environment_status.isin(STATUS_VALUES).all():
        raise ValueError('Stato ambiente non valido; usare: ' + ', '.join(sorted(STATUS_VALUES)))
    if not results.cve_id.isin(candidates.cve_id).all():
        raise ValueError('Il file esiti contiene record estranei al perimetro corrente.')
    note = results.evidence_note.fillna('').astype(str).str.strip()
    if (results.environment_status.eq('confirmed') & note.eq('')).any():
        raise ValueError('Ogni esito confirmed richiede una nota di evidenza.')
    log = results.set_index('cve_id')
    candidates['kernjc_environment_status'] = candidates.cve_id.map(log.environment_status).fillna('not_tested')
    candidates['kernjc_environment_evidence'] = candidates.cve_id.map(log.evidence_note).fillna('')
    confirmed = candidates.kernjc_environment_status.eq('confirmed')

    # Riutilizzo esatto della formula v4.4; sostituisce, non somma, il contributo Vulhub.
    components = score_components(candidates)
    candidates['score_original'] = components.total
    candidates['score_environment'] = confirmed.astype(int)*WEIGHTS['vulhub']
    candidates['score_exploit'] = components[['msf','edb','poc']].sum(axis=1).round(2)
    candidates['score_detection'] = components.detection
    candidates['score_kernel_provisional'] = (
        components[['msf','edb','poc','detection']].sum(axis=1)
        + candidates.score_environment
    ).round(2)
    candidates['benchmark_eligible'] = confirmed & candidates.score_kernel_provisional.ge(MIN_SCORE)
    assert candidates.score_kernel_provisional.between(0,100).all()
    assert candidates.score_environment.eq(30*confirmed.astype(int)).all()
    assert not candidates.loc[~confirmed,'benchmark_eligible'].any()
    return candidates

if ENV_RESULTS_CSV.is_file():
    environment_log = pd.read_csv(ENV_RESULTS_CSV, dtype=str)
    print('Esiti locali importati:', len(environment_log))
else:
    environment_log = pd.DataFrame(columns=LOG_COLUMNS)
    print('Nessun file di esiti locali: tutti non testati.')

kernel_evaluation = apply_kernel_scoring(eligible, environment_log)
status_summary = kernel_evaluation.kernjc_environment_status.value_counts().reindex(sorted(STATUS_VALUES),fill_value=0)
print('CVE nel perimetro documentale:', len(kernel_evaluation))
print('Stati ambiente:')
print(status_summary.to_string())
print('Con ambiente confermato (+30):', int(kernel_evaluation.score_environment.eq(30).sum()))
print('Con ambiente confermato e score >=',MIN_SCORE,':', int(kernel_evaluation.benchmark_eligible.sum()))
if not status_summary['confirmed']:
    print('Nessun ambiente confermato: il benchmark definitivo non è ancora popolato.')

candidates_csv = Path('kernel_candidates.csv').resolve()
kernel_evaluation.to_csv(candidates_csv, index=False)
print(f'CSV esportato: {candidates_csv} ({len(kernel_evaluation)} righe)')

## 9. Controlli della nuova politica di scoring

Controlli su record sintetici, senza attività di laboratorio: il paper non modifica il punteggio ambiente; una conferma documentata vale esattamente 30; uno stato fallito o inconclusivo non vale 30; la soglia è inclusiva; senza conferma il benchmark non viene popolato.

In [10]:
# Dati sintetici esclusivamente per verificare la logica del calcolo.
synthetic = pd.DataFrame({
    'cve_id':['TEST-1','TEST-2','TEST-3','TEST-4','TEST-5'],
    'documented_evidence':[True]*5,
    'kernjc_paper_match':[True,False,True,False,False],
    'has_vulhub':[False]*5,
    'msf_compatible':[False]*5,
    'edb_compatible':[True,True,True,True,False],
    'has_poc_reference':[False,False,False,False,True],
    'edb_verified':[True,True,True,True,False],
    'msf_best_rank':[float('nan')]*5,
    'has_detection_rule':[False]*5
})
synthetic_log = pd.DataFrame([
    ['TEST-2','confirmed','Evidenza sintetica'],
    ['TEST-3','failed','Evidenza sintetica'],
    ['TEST-4','inconclusive','Evidenza sintetica'],
    ['TEST-5','confirmed','Evidenza sintetica'],
],columns=LOG_COLUMNS)
check = apply_kernel_scoring(synthetic, synthetic_log)
assert check.score_environment.tolist() == [0,30,0,0,30]
assert check.score_kernel_provisional.tolist() == [30,60,30,30,50]
# L'asserzione sulla soglia 60 è riferita alla configurazione di riferimento.
assert check.benchmark_eligible.tolist() == [False, 60 >= MIN_SCORE, False, False, 50 >= MIN_SCORE]
assert check.kernjc_environment_status.tolist() == ['not_tested','confirmed','failed','inconclusive','confirmed']
print('Controlli scoring kernel: OK.')

Controlli scoring kernel: OK.


## 10. Interpretazione

- Il gruppo in comune col paper rimane un riferimento di letteratura.
- Il perimetro documentale è distinto dal benchmark definitivo.
- Il risultato dell'audit resta riproducibile anche quando non è stato ancora eseguito alcun test locale.
- Gli stati locali vanno raccolti in un laboratorio autorizzato e isolato; la presenza dell'ambiente non dimostra da sola la riproduzione della vulnerabilità o l'adeguatezza della telemetria.
- I contributi originali di Metasploit, Exploit-DB, PoC, Falco, Sigma ed Elastic sono invariati. Per il trasferimento dei 30 punti ambiente si usa KernJC al posto di Vulhub.